[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefHosni/prompt-engineering-book-2e/blob/main/notebooks/ch04_room_to_think.ipynb)

# Chapter 4 — Giving the Model Room to Think

The headline finding in this chapter's first draft was an artifact of the
author's own prompt. "Reply with only the number" suppressed the working and
produced a 2/12 baseline that had nothing to do with reasoning.

Both conditions are run here, because the difference between them *is* the
chapter.

In [1]:
# Colab only: fetch the shared helpers and install the SDKs.
# Running locally? Skip this cell -- you already have them.
import os, sys
if "google.colab" in sys.modules:
    !pip -q install anthropic openai python-dotenv
    !git clone -q https://github.com/youssefHosni/prompt-engineering-book-2e /content/peb-book
    sys.path.insert(0, "/content/peb-book")
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")


In [2]:
import os
import re

from peb import ask

RUNS = 12

# The chapter measures TWO models, and the contrast is the whole point:
#
#   claude-haiku-4.5   12/12 normally, 2/12 with "Reply with only the number"
#   claude-sonnet-5    12/12 either way -- its thinking does not live in the text
#
# Running only a reasoning model gives 12/12 in both arms and looks like a null
# result. That is not a refutation of the chapter, it is the chapter's second
# row. Both are run here.
MODELS = os.getenv("PEB_MODEL", "claude-haiku-4.5,claude-sonnet-5").split(",")

## §4.2 Four numbered steps in one pass

In [3]:
EMAIL = """Hi - the Nordic 400 I ordered last month has started making a loud
grinding noise whenever I pull a shot, and the crema has gone. It was fine for
three weeks. Is this something I can fix or do I need to send it back?"""

steps = """For the customer email below, do the following in order:

1. Identify the product the customer is writing about.
2. Decide whether the message is a complaint, a question, or praise.
3. If it is a complaint, name the specific fault.
4. Draft a two-sentence reply.

Return each step on its own line, labelled.

Email: ```{email}```"""

text, _ = ask(steps.format(email=EMAIL), max_tokens=4000)
print(text.strip()[:700])
print(f"\nsteps present: {sum(text.count(f'{i}.') > 0 for i in range(1, 5))}/4")

1. Product: The Nordic 400 espresso machine, purchased last month.

2. Category: Complaint (with an embedded question about repair vs. return).

3. Specific fault: A loud grinding noise during shot extraction accompanied by loss of crema, developing after roughly three weeks of normal operation — symptoms consistent with a pump or grinder/burr problem rather than user error.

4. Draft reply:
"Thanks for getting in touch, and I'm sorry your Nordic 400 has developed a grinding noise and lost its crema — that combination usually points to a pump or grinder fault rather than anything you've done, so please stop using the machine for now. As it's under a month old it's covered by warranty: reply 

steps present: 4/4


## §4.4 The instruction that suppresses the working

One variable changes between these two conditions: whether the model is
allowed to show its steps. Measure the prompt you wrote, not the one you
meant to write.

In [4]:
# The original problem here was 3x12 + 2x18 - 7. Both conditions scored 12/12
# on it: a current model does that arithmetic without needing to write anything
# down, so the prompt difference had nothing to bite on. An experiment where
# both arms score full marks measures nothing -- §14.1, again.
#
# This one has multiple dependent steps and a unit change, which is where
# suppressing the working actually costs something.
WORD_PROBLEM = (
    "A depot receives 7 pallets. Each pallet holds 24 cases; each case holds 18 "
    "tins. During unloading 3 cases are crushed, and a further 2 pallets are "
    "found to be half-empty (12 cases each instead of 24). Tins are then packed "
    "into cartons of 45 for dispatch. How many full cartons can be dispatched, "
    "and how many tins are left over?")
# 5 full pallets x 24 = 120 cases, plus 2 x 12 = 24 cases -> 144, minus 3 = 141
# 141 x 18 = 2538 tins -> 2538 / 45 = 56 cartons remainder 18
ANSWER_CARTONS, ANSWER_LEFT = 56, 18

for model in MODELS:
    print(f"\n{model}")
    for label, suffix in (
            ("room to think", " Show your working, then give the answer as "
                              "'CARTONS: <n>  LEFTOVER: <n>'."),
            ("answer only",   " Reply with only 'CARTONS: <n>  LEFTOVER: <n>' and "
                              "no working.")):
        right = tokens = 0
        for _ in range(RUNS):
            text, usage = ask(WORD_PROBLEM + suffix, model=model, max_tokens=4000)
            tokens += usage.output_tokens
            m = re.search(r"CARTONS:\s*(\d+)\s+LEFTOVER:\s*(\d+)", text.upper())
            right += bool(m) and (int(m.group(1)), int(m.group(2))) == (ANSWER_CARTONS,
                                                                        ANSWER_LEFT)
        print(f"  {label:<15} {right}/{RUNS} correct   "
              f"{tokens / RUNS:.0f} output tokens/problem")


claude-haiku-4.5


  room to think   12/12 correct   261 output tokens/problem


  answer only     0/12 correct   16 output tokens/problem

claude-sonnet-5


  room to think   12/12 correct   562 output tokens/problem


  answer only     12/12 correct   175 output tokens/problem
